# J1S3 — EDA & Plotly : Exploration Visuelle des Données Credit Risk
## Formation BankRisk Intelligence Platform · Jour 1 · Session 3

**Durée :** 13h00 – 16h30  
**Prérequis :** J1S2 complétée — `credit_features_j1.parquet` **(15 colonnes, sans loan_grade)** dans `data/processed/`  
**Livrable :** 9 figures interactives HTML dans `figures/`

---

### Contexte

On repart du fichier `credit_features_j1.parquet` produit en J1S2.  
L'objectif est de **transformer les analyses numériques de J1S2 en visualisations** qui justifient nos choix de features et sont communicables à un directeur de risque.

**Ce que J1S2 a calculé → J1S3 va visualiser :**
- RENT = 31,6 % · MORTGAGE = 12,6 % → box plot + bar chart home_ownership
- `debt_service_rate` Spearman=+0,389 → heatmap + scatter
- Rupture Q1→Q2 : 39,7 % → 21,3 % (−18,6 pts) → bar chart quartiles
- DEBTCONS=28,6 % · MEDICAL=26,7 % → validation `high_risk_intent`

**Fil rouge :** les insights visuels justifient les 4 features créées en J1S2  
et annoncent le modèle RF Baseline (AUC=0,929, seuil t=0,42) de J2S2.

> ✓ `loan_grade` est absente du parquet — aucun graphique ne la référence.  
> ✓ `loan_int_rate` est présente comme variable source réhabilitée (Spearman=+0,298).

*Dataset : credit_risk_dataset.csv — 32 581 lignes · CC0 · Taux de défaut : 21,8 %*

---
## Setup — Installation & Imports

In [9]:
# ============================================================
# CELL 0 — Setup : ROOT · pip install · imports
# NE PAS DIVISER — ROOT doit être défini avant pip install
# ============================================================
import sys, subprocess
from pathlib import Path

# ── Détection environnement ──────────────────────────────────────────────
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    ROOT = Path("/content")
    print("✓ Google Colab")
else:
    ROOT = Path().resolve()
    while ROOT != ROOT.parent:
        if (ROOT / "data").exists():
            break
        ROOT = ROOT.parent
    print(f"✓ VS Code — ROOT = {ROOT}")

(ROOT / "figures").mkdir(exist_ok=True)

# ── pip install ──────────────────────────────────────────────────────────
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "pandas>=2.0", "numpy>=1.24", "pyarrow>=12.0", "plotly>=5.0"], check=True)
print("✓ Packages installés")

# ── Imports ──────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print(f"pandas {pd.__version__} · numpy {np.__version__} · plotly {plotly.__version__}")

# ── Vérification parquet J1S2 ────────────────────────────────────────────
PARQUET_PATH = ROOT / "data" / "processed" / "credit_features_j1.parquet"
assert PARQUET_PATH.exists(), f"❌ Parquet introuvable : {PARQUET_PATH} — relancer J1S2 d'abord."
print(f"✓ Parquet J1S2 présent : {PARQUET_PATH}")

✓ VS Code — ROOT = D:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation
✓ Packages installés
pandas 2.3.3 · numpy 2.2.6 · plotly 6.8.0
✓ Parquet J1S2 présent : D:\FORMATION\PTYHON FOR DATA\files\Jour1\session1\workspace\credit-risk-formation\data\processed\credit_features_j1.parquet


In [10]:
# ── Chargement et vérifications ──────────────────────────────────────────
df = pd.read_parquet(PARQUET_PATH)

print(f"Shape : {df.shape}")                              # → (32581, 15)
print(f"Taux de défaut : {df['loan_status'].mean():.1%}")  # → 21.8%
print("\nColonnes :")
print(df.columns.tolist())

# ── Assertions de cohérence avec J1S2 ────────────────────────────────────
assert df.shape == (32581, 15), f"❌ Shape inattendu : {df.shape} — attendu (32581, 15)"
assert abs(df['loan_status'].mean() - 0.218) < 0.002, "❌ Taux de défaut incorrect"

# Variables absentes — loan_grade et ses dérivés ne doivent pas être là
assert 'loan_grade' not in df.columns,           "❌ loan_grade présente — relancer J1S2"
assert 'loan_grade_enc' not in df.columns,       "❌ loan_grade_enc présente"
assert 'grade_pct_interaction' not in df.columns, "❌ grade_pct_interaction présente"

# 4 features pipeline J1S2 doivent être présentes
assert 'debt_service_rate' in df.columns,     "❌ debt_service_rate manquante"
assert 'monthly_payment_proxy' in df.columns, "❌ monthly_payment_proxy manquante"
assert 'log_income' in df.columns,            "❌ log_income manquante"
assert 'high_risk_intent' in df.columns,      "❌ high_risk_intent manquante"
assert 'loan_int_rate' in df.columns,         "❌ loan_int_rate manquante (réhabilitée)"

print("\n✅ Toutes les vérifications passées — pipeline 100% sans loan_grade")

Shape : (32581, 15)
Taux de défaut : 21.8%

Colonnes :
['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'debt_service_rate', 'monthly_payment_proxy', 'log_income', 'high_risk_intent']

✅ Toutes les vérifications passées — pipeline 100% sans loan_grade


### Interprétation du chargement

- `df.shape = (32581, 15)` : 11 colonnes sources (sans `loan_grade`) + 4 features J1S2
- `loan_status.mean() = 0.218` : **21,8 %** de défaut — référence pour toute l'EDA
- Tout segment avec un taux > 21,8 % est en **surrisque** vs la moyenne du portefeuille

**4 features pipeline présentes — sans dépendance externe :**

| Feature | Construction | Spearman |
|---------|-------------|----------|
| `debt_service_rate` | loan_int_rate × loan_percent_income | **+0,389** |
| `monthly_payment_proxy` | loan_amnt / (income/12) | +0,322 |
| `log_income` | np.log1p(person_income) | −0,272 |
| `high_risk_intent` | flag DEBTCONS ou MEDICAL | +0,101 |

---
## Bloc 1 — Distributions : Histogrammes & Box Plots
**Durée : 45 min**

Objectif : visualiser la distribution des revenus et du ratio prêt/revenu pour comprendre
pourquoi `log_income` et `debt_service_rate` ont été créés.

### 1.1 Histogramme — Distribution des revenus par statut de prêt

In [24]:
# ── Vue 1 : zoom P99 ─────────────────────────────────────────────────────
# ⚠️ JAMAIS log_x=True : Plotly calcule les bins en espace linéaire puis applique
# l'axe log → barres vides ou effondrées. Toujours range_x ou np.log1p().

p99 = df['person_income'].quantile(0.99)
p25 = df['person_income'].quantile(0.25)
p75 = df['person_income'].quantile(0.75)
print(f"P25 : {p25:,.0f} $  |  Médiane : {df['person_income'].median():,.0f} $  |  P75 : {p75:,.0f} $")
print(f"P99 : {p99:,.0f} $  |  Max brut : {df['person_income'].max():,.0f} $ (outlier → cappé J1S4)")

fig1 = px.histogram(
    df, x='person_income', color='loan_status',
    barmode='overlay', nbins=80, range_x=[0, p99],
    color_discrete_map={0: '#1C7293', 1: '#02C39A'},
    labels={'loan_status': 'Défaut', 'person_income': 'Revenu annuel ($)'},
    title=f'Distribution des revenus — zoom P99 ({p99/1000:.0f}k$) · 32 581 dossiers'
)
fig1.add_vline(x=p25, line_dash='dot', line_color='#FFA07A',
               annotation_text=f'P25={p25/1000:.0f}k$', annotation_position='top right')
fig1.add_vline(x=p75, line_dash='dot', line_color='#FFA07A',
               annotation_text=f'P75={p75/1000:.0f}k$', annotation_position='top right')
fig1.update_layout(bargap=0.02, plot_bgcolor='#021B2E',
                   paper_bgcolor='#021B2E', font_color='white')
fig1.write_html(str(ROOT / 'figures' / 'hist_income.html'))
fig1.show()
print("✅ figures/hist_income.html sauvegardé")
print("→ Les défauts (vert) se concentrent sur les revenus < 40 000 $ — justifie log_income")

P25 : 38,500 $  |  Médiane : 55,000 $  |  P75 : 79,200 $
P99 : 225,200 $  |  Max brut : 6,000,000 $ (outlier → cappé J1S4)


✅ figures/hist_income.html sauvegardé
→ Les défauts (vert) se concentrent sur les revenus < 40 000 $ — justifie log_income


In [12]:
# ── Vue 2 : log1p ────────────────────────────────────────────────────────
# log_income est déjà dans le parquet J1S2 (np.log1p appliqué en J1S2)
# On l'utilise directement — pas besoin de recalculer

print(f"Skewness person_income : {df['person_income'].skew():.2f}")
print(f"Skewness log_income    : {df['log_income'].skew():.2f}")
print("→ La normalisation log1p réduit drastiquement l'asymétrie")

fig1b = px.histogram(
    df, x='log_income', color='loan_status',
    barmode='overlay', nbins=80,
    color_discrete_map={0: '#1C7293', 1: '#02C39A'},
    labels={'loan_status': 'Défaut', 'log_income': 'log(1 + revenu annuel)'},
    title='Distribution log_income — feature J1S2 · tous les dossiers · outliers intégrés'
)
fig1b.update_layout(bargap=0.02, plot_bgcolor='#021B2E',
                    paper_bgcolor='#021B2E', font_color='white')
fig1b.write_html(str(ROOT / 'figures' / 'hist_income_log.html'))
fig1b.show()
print("✅ figures/hist_income_log.html sauvegardé")

Skewness person_income : 32.87
Skewness log_income    : 0.16
→ La normalisation log1p réduit drastiquement l'asymétrie


✅ figures/hist_income_log.html sauvegardé


**⚠️ Piège classique : `log_x=True`**

`log_x=True` dans `px.histogram` calcule les bins en espace **linéaire** puis applique l'axe log → barres vides.  
**Solution correcte :** `np.log1p()` appliqué sur la colonne **avant** Plotly — c'est exactement ce que fait `log_income` créé en J1S2.  
La colonne est déjà dans le parquet : on la passe directement à Plotly.

### 1.2 Box Plot — loan_percent_income par home_ownership et statut

In [13]:
# Box plot home_ownership × loan_percent_income
# Sans loan_grade, home_ownership est le premier discriminant catégoriel (Spearman=+0.224)
# RENT=31.6% vs OWN=7.5% → écart de 24 pts

fig2 = px.box(
    df, x='person_home_ownership', y='loan_percent_income',
    color='loan_status',
    color_discrete_map={0: '#065A82', 1: '#02C39A'},
    category_orders={'person_home_ownership': ['OWN', 'MORTGAGE', 'OTHER', 'RENT']},
    labels={'loan_status': 'Défaut', 'loan_percent_income': 'Ratio prêt/revenu',
            'person_home_ownership': 'Type de logement'},
    title='Ratio prêt/revenu par type de logement — RENT cumule deux facteurs de risque'
)
fig2.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E', font_color='white')
fig2.write_html(str(ROOT / 'figures' / 'box_home.html'))
fig2.show()
print("✅ figures/box_home.html sauvegardé")
print("→ RENT : boîte haute (ratio élevé) ET taux de défaut 31.6% — double pression")
print("→ OWN : boîte basse (ratio faible) ET taux de défaut 7.5% — profil sûr")
print("→ Justifie debt_service_rate qui combine loan_int_rate × loan_percent_income")

✅ figures/box_home.html sauvegardé
→ RENT : boîte haute (ratio élevé) ET taux de défaut 31.6% — double pression
→ OWN : boîte basse (ratio faible) ET taux de défaut 7.5% — profil sûr
→ Justifie debt_service_rate qui combine loan_int_rate × loan_percent_income


**Ce que vous voyez :**
- RENT : boîtes vertes (défaut) systématiquement **au-dessus** des bleues — ratio plus élevé
- OWN : boîtes très basses (< 0,10) — propriétaires bien couverts
- MORTGAGE : position intermédiaire — hypothèque = engagement financier structuré
- Sans `loan_grade`, `person_home_ownership` est le premier discriminant catégoriel (Spearman=+0,224)

---
## Bloc 2 — Corrélations : Heatmap & Scatter Matrix
**Durée : 50 min**

Objectif : confirmer visuellement les classements Spearman calculés en J1S2.  
`debt_service_rate` doit apparaître comme le signal le plus fort (+0,389).  
`high_risk_intent` doit apparaître orthogonal (r≈0,002).

### 2.1 Heatmap de corrélations Spearman

In [14]:
# Heatmap Spearman — variables numériques CONTINUES uniquement
# Spearman n'est légitime qu'entre variables continues/ordinales.
# → on retire les binaires (high_risk_intent, loan_status) : elles seront
#   mesurées autrement (taux de défaut par groupe, cf. 2.1ter et Bloc 3).

num_cols = [
    'person_income', 'loan_amnt', 'loan_int_rate',
    'loan_percent_income', 'debt_service_rate',
    'monthly_payment_proxy', 'log_income',
]

corr = df[num_cols].corr(method='spearman').round(2)

fig3 = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    text_auto=True,
    title='Corrélations Spearman — variables numériques continues du pipeline'
)
fig3.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E', font_color='white')
fig3.write_html(str(ROOT / 'figures' / 'heatmap_corr.html'))
fig3.show()
print("✅ figures/heatmap_corr.html sauvegardé")
print("\nAssociations Spearman entre variables continues (extraits) :")
print(f"  debt_service_rate  ↔ monthly_payment_proxy : {corr.loc['debt_service_rate','monthly_payment_proxy']:+.2f}")
print(f"  debt_service_rate  ↔ loan_percent_income   : {corr.loc['debt_service_rate','loan_percent_income']:+.2f}")
print(f"  log_income         ↔ person_income         : {corr.loc['log_income','person_income']:+.2f}")
print("\n⚠️  loan_status et high_risk_intent ne sont PAS dans cette matrice :")
print("    ce sont des variables binaires → association mesurée par le")
print("    taux de défaut par groupe (voir 2.1ter et Bloc 3.1), pas par Spearman.")

✅ figures/heatmap_corr.html sauvegardé

Corrélations Spearman vs loan_status :
  person_income                : -0.270
  loan_amnt                    : +0.080
  loan_int_rate                : +0.320 ← réhabilité (sans grade)
  loan_percent_income          : +0.320
  debt_service_rate            : +0.390 ← signal #1
  monthly_payment_proxy        : +0.320
  log_income                   : -0.270
  high_risk_intent             : +0.100 ← orthogonal ✓


**Lecture de la heatmap :**

Cette matrice ne contient **que des variables numériques continues** — c'est la seule situation où Spearman est légitime. L'association avec la cible `loan_status` (binaire) se lit ailleurs, via le **taux de défaut par groupe** (cf. 2.1ter et Bloc 3.1).

| Variable continue | Spearman vs `loan_status`* | Interprétation |
|-------------------|:--:|----------------|
| `debt_service_rate` | **+0,389** | Feature #1 — taux × ratio |
| `monthly_payment_proxy` | **+0,322** | Charge mensuelle relative |
| `loan_percent_income` | +0,316 | Signal source |
| `loan_int_rate` | +0,298 | Réhabilité sans grade |
| `log_income` / `person_income` | −0,272 | Revenu protège |

\*Ces valeurs restent les repères connus du pipeline : elles décrivent la **force du signal continu** vers le défaut. Le calcul rigoureux d'une association continu ↔ binaire relève d'une autre mesure (formalisée en J2S2).

**Rouge** = association positive (→ défaut). **Bleu** = association négative (→ non-défaut).

**Et `high_risk_intent` ?** C'est une variable **binaire** : son apport ne se lit **pas** sur cette heatmap Spearman. On le mesure via le **taux de défaut par groupe** (Bloc 4.2), qui révèle son caractère orthogonal — un apport indépendant des autres signaux.

### 2.1bis — Pourquoi une seule heatmap ne suffit pas

> **💡 À retenir : la mesure d'association dépend du *type* des variables croisées.**
>
> Une heatmap Spearman ne sait comparer que des nombres qui se classent (continus/ordinaux). Dès qu'une variable est **catégorielle** (motif du prêt `loan_intent`, type de logement `person_home_ownership`) ou **binaire** (défaut `loan_status`, flag `high_risk_intent`), Spearman n'a **pas de sens** : ranger « LOCATION » avant « PROPRIÉTAIRE » ne veut rien dire. Il faut alors une **autre mesure**, adaptée au couple de types.

**Mini-tableau de routage (version débutant) :**

| Croisement | Mesure d'association | Où en J1S3 ? |
|------------|----------------------|--------------|
| num ↔ num | **Spearman** (heatmap 2.1) | ✅ ici |
| catégorielle → cible | **Taux de défaut par catégorie** (barplot) | ✅ 2.1ter & Bloc 3 |
| catégorielle ↔ catégorielle | mesure dédiée | 🔜 **J2S2** |
| num ↔ catégorielle | mesure dédiée | 🔜 **J2S2** |

Les mesures formelles (V de Cramér, rapport de corrélation η, VIF) seront **vues en détail en J2S2**. En J1S3, on retient juste l'idée : *une seule heatmap ne peut pas tout capturer.*

### 2.1ter — Lire l'association d'une variable qualitative avec la cible

Pour une variable **catégorielle**, la bonne lecture n'est pas un coefficient de Spearman mais le **taux de défaut par catégorie** : si le taux varie fortement d'une catégorie à l'autre, la variable *porte de l'information* sur le défaut. Illustrons-le sur `loan_intent` (motif du prêt).

In [ ]:
# Association d'une variable QUALITATIVE avec la cible : le taux de défaut par catégorie
# Ceci EST la mesure d'association catégorielle → cible (pas Spearman).
# Même logique que le Bloc 3.1 (taux de défaut par home_ownership).

default_intent = (
    df.groupby('loan_intent')['loan_status']
    .agg(taux_defaut='mean', nb_dossiers='count')
    .reset_index()
    .sort_values('taux_defaut', ascending=False)
)

fig_intent_assoc = px.bar(
    default_intent,
    x='loan_intent', y='taux_defaut',
    color='taux_defaut',
    color_continuous_scale=['#065A82', '#1C7293', '#FFA07A', '#02C39A'],
    text=default_intent['taux_defaut'].map('{:.1%}'.format),
    labels={'taux_defaut': 'Taux de défaut', 'loan_intent': 'Motif du prêt'},
    title="Taux de défaut par motif du prêt — l'association d'une variable qualitative se lit ainsi"
)
fig_intent_assoc.update_traces(textposition='outside')
fig_intent_assoc.add_hline(y=0.218, line_dash='dot', line_color='white',
                           annotation_text='Moyenne 21,8 %', annotation_position='top right')
fig_intent_assoc.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E',
                               font_color='white', yaxis_tickformat='.0%',
                               xaxis_tickangle=-30)
fig_intent_assoc.show()

print("Taux de défaut par motif du prêt :")
for _, row in default_intent.iterrows():
    flag = ' ← surrisque' if row['taux_defaut'] > 0.218 else ' ← sous la moyenne'
    print(f"  {row['loan_intent']:18s} : {row['taux_defaut']:.1%}  ({row['nb_dossiers']:>5} dossiers){flag}")

print("\n➡️  Le taux varie nettement selon le motif : loan_intent PORTE de l'information.")
print("    C'EST votre mesure d'association pour une variable qualitative → cible,")
print("    exactement comme le taux de défaut par home_ownership au Bloc 3.1.")

### 2.2 Scatter Matrix — 4 features vs défaut

In [15]:
# Scatter matrix — 4 features pipeline
# ⚠️ 32 581 points = navigateur bloqué. sample(3000) suffit.
# random_state=42 garantit la reproductibilité (tous voient le même graphique)

fig4 = px.scatter_matrix(
    df.sample(3000, random_state=42),
    dimensions=['debt_service_rate', 'monthly_payment_proxy',
                'loan_percent_income', 'log_income'],
    color='loan_status',
    color_discrete_map={0: '#1C7293', 1: '#02C39A'},
    opacity=0.4,
    labels={'loan_status': 'Défaut'},
    title='Scatter matrix — 4 features pipeline vs défaut (échantillon 3 000 dossiers)'
)
fig4.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E', font_color='white')
fig4.write_html(str(ROOT / 'figures' / 'scatter_matrix.html'))
fig4.show()
print("✅ figures/scatter_matrix.html sauvegardé")
print("→ Sur l'axe debt_service_rate : points verts (défauts) nettement décalés vers la droite")
print("→ C'est la séparation que RF Baseline (AUC=0.929) va apprendre")

✅ figures/scatter_matrix.html sauvegardé
→ Sur l'axe debt_service_rate : points verts (défauts) nettement décalés vers la droite
→ C'est la séparation que RF Baseline (AUC=0.929) va apprendre


---
## Bloc 3 — Analyse Bivariée : Risque par Segment
**Durée : 55 min**

Objectif : visualiser le gradient de risque par home_ownership et l'interaction entre les deux features principales.

### 3.1 Taux de défaut par home_ownership

In [16]:
# Taux de défaut par type de logement
# Sans loan_grade, home_ownership devient le premier discriminant catégoriel visible
# RENT=31.6% vs OWN=7.5% : écart de 24 pts

default_home = (
    df.groupby('person_home_ownership')['loan_status']
    .agg(taux_defaut='mean', nb_dossiers='count')
    .reset_index()
    .sort_values('taux_defaut', ascending=False)
)

fig5 = px.bar(
    default_home,
    x='person_home_ownership', y='taux_defaut',
    color='taux_defaut',
    color_continuous_scale=['#065A82', '#1C7293', '#FFA07A', '#02C39A'],
    text=default_home['taux_defaut'].map('{:.1%}'.format),
    labels={'taux_defaut': 'Taux de défaut', 'person_home_ownership': 'Type de logement'},
    title='Taux de défaut par type de logement — gradient RENT → OWN'
)
fig5.update_traces(textposition='outside')
fig5.add_hline(y=0.218, line_dash='dot', line_color='white',
               annotation_text='Moyenne 21,8 %', annotation_position='top right')
fig5.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E',
                   font_color='white', yaxis_tickformat='.0%')
fig5.write_html(str(ROOT / 'figures' / 'defaut_home.html'))
fig5.show()
print("✅ figures/defaut_home.html sauvegardé")
print("Taux de défaut par type de logement :")
for _, row in default_home.iterrows():
    flag = ' ← surrisque' if row['taux_defaut'] > 0.218 else ' ← sous la moyenne'
    print(f"  {row['person_home_ownership']:10s} : {row['taux_defaut']:.1%}  ({row['nb_dossiers']:>5} dossiers){flag}")

✅ figures/defaut_home.html sauvegardé
Taux de défaut par type de logement :
  RENT       : 31.6%  (16446 dossiers) ← surrisque
  OTHER      : 30.8%  (  107 dossiers) ← surrisque
  MORTGAGE   : 12.6%  (13444 dossiers) ← sous la moyenne
  OWN        : 7.5%  ( 2584 dossiers) ← sous la moyenne


**Ce que ce graphique communique à un directeur de risque :**

RENT et OTHER sont au-dessus de la moyenne (21,8 %) → surrisque.  
MORTGAGE et OWN sont en dessous → profil protecteur.

Question : une règle simple « RENT = majoration de taux » serait-elle suffisante ?  
Réfléchissez à ce que RF (AUC=0,929) apporte au-delà — il combine `person_home_ownership` avec `debt_service_rate`, `monthly_payment_proxy` et les 2 autres features pour un scoring nuancé.

### 3.2 Scatter bivarié — debt_service_rate vs monthly_payment_proxy

In [17]:
# Scatter bivarié : les deux features les plus importantes du pipeline
# debt_service_rate (Spearman=0.389) vs monthly_payment_proxy (importance GB=28.6%)
# Coloré par home_ownership + symbole par statut de défaut

fig6 = px.scatter(
    df.sample(5000, random_state=42),
    x='debt_service_rate',
    y='monthly_payment_proxy',
    color='person_home_ownership',
    symbol='loan_status',
    opacity=0.5,
    color_discrete_map={
        'RENT': '#FF6B6B', 'MORTGAGE': '#1C7293',
        'OWN': '#02C39A', 'OTHER': '#FFA07A'
    },
    labels={
        'debt_service_rate': 'debt_service_rate (taux × ratio)',
        'monthly_payment_proxy': 'monthly_payment_proxy (charge mensuelle)',
        'loan_status': 'Défaut', 'person_home_ownership': 'Logement'
    },
    title='Deux features pipeline par home_ownership et statut — complémentarité des signaux'
)
fig6.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E', font_color='white')
fig6.write_html(str(ROOT / 'figures' / 'scatter_features.html'))
fig6.show()
print("✅ figures/scatter_features.html sauvegardé")
print("→ RENT (rouge) : concentrés en haut à droite — double pression taux + charge")
print("→ OWN (vert) : concentrés en bas à gauche — profil sûr sur les deux axes")
print("→ La frontière non-linéaire justifie RF plutôt qu'une règle manuelle")

✅ figures/scatter_features.html sauvegardé
→ RENT (rouge) : concentrés en haut à droite — double pression taux + charge
→ OWN (vert) : concentrés en bas à gauche — profil sûr sur les deux axes
→ La frontière non-linéaire justifie RF plutôt qu'une règle manuelle


**Deux features complémentaires, pas redondantes :**

- `debt_service_rate` = loan_int_rate × loan_percent_income → **pression financière combinée** (taux + ratio)
- `monthly_payment_proxy` = loan_amnt / (income/12) → **charge mensuelle absolue**

Leur corrélation est ~0,57 — corrélées mais distinctes. La frontière de séparation n'est pas une droite simple → justifie RF (modèle non-linéaire, AUC=0,929 vs LR AUC=0,866).

---
## Bloc 4 — Fil Rouge : Features Pipeline → Modèle
**Durée : 40 min**

Objectif : confirmer visuellement que chaque feature J1S2 apporte un signal discriminant.  
Chaque graphique doit répondre à la question : *pourquoi cette feature est-elle dans le modèle ?*

### 4.1 Rupture Q1→Q2 — justification de log_income

In [18]:
# Quartiles de revenu — la rupture Q1→Q2 en image
# Calculé ici pour la visualisation (variable temporaire)
df['_income_q'] = pd.qcut(df['person_income'], q=4, labels=['Q1','Q2','Q3','Q4'])

default_quartile = (
    df.groupby('_income_q', observed=True)['loan_status']
    .agg(taux_defaut='mean', nb_dossiers='count')
    .reset_index()
)

fig7 = px.bar(
    default_quartile,
    x='_income_q', y='taux_defaut',
    text=default_quartile['taux_defaut'].map('{:.1%}'.format),
    color_discrete_sequence=['#1C7293'],
    labels={'_income_q': 'Quartile de revenu', 'taux_defaut': 'Taux de défaut'},
    title='Taux de défaut par quartile de revenu — rupture Q1→Q2 de −18,6 pts'
)
fig7.update_traces(textposition='outside')
fig7.add_hline(y=0.218, line_dash='dot', line_color='white',
               annotation_text='Moyenne 21,8 %', annotation_position='top right')
fig7.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E',
                   font_color='white', yaxis_tickformat='.0%')
fig7.write_html(str(ROOT / 'figures' / 'defaut_quartile.html'))
fig7.show()
df.drop(columns=['_income_q'], inplace=True)
print("✅ figures/defaut_quartile.html sauvegardé")
print("→ Q1 : 39.7%  →  Q2 : 21.3%  →  Q3 : 17.1%  →  Q4 : 9.1%")
print("→ Chute Q1→Q2 = −18.6 pts — non-linéarité → justifie np.log1p (log_income J1S2)")

✅ figures/defaut_quartile.html sauvegardé
→ Q1 : 39.7%  →  Q2 : 21.3%  →  Q3 : 17.1%  →  Q4 : 9.1%
→ Chute Q1→Q2 = −18.6 pts — non-linéarité → justifie np.log1p (log_income J1S2)


**La rupture Q1→Q2 en image :**

- Q1 (revenus < ~38 500 $) : **39,7 %** de défaut
- Q2 (~38 500 $ – 60 000 $) : **21,3 %** → **−18,6 pts en un seul saut de quartile**
- Q4 (revenus > ~79 200 $) : < 10 %

Cette non-linéarité est la raison pour laquelle `log_income = np.log1p(person_income)` a été créée en J1S2. La régression logistique (J2S2) modélise mieux une relation avec la transformation logarithmique qu'avec le revenu brut.

### 4.2 Signal high_risk_intent — DEBTCONSOLIDATION & MEDICAL

In [19]:
# Taux de défaut par intention — justifie le flag high_risk_intent
default_intent = (
    df.groupby('loan_intent')['loan_status']
    .agg(taux_defaut='mean', nb_dossiers='count')
    .reset_index()
    .sort_values('taux_defaut', ascending=False)
)

# Couleur : orange pour les high_risk, teal pour les autres
color_map = {
    'DEBTCONSOLIDATION': '#FFA07A', 'MEDICAL': '#FFA07A',
    'HOMEIMPROVEMENT': '#1C7293', 'PERSONAL': '#1C7293',
    'EDUCATION': '#1C7293', 'VENTURE': '#065A82'
}

fig8 = px.bar(
    default_intent,
    x='loan_intent', y='taux_defaut',
    text=default_intent['taux_defaut'].map('{:.1%}'.format),
    color='loan_intent',
    color_discrete_map=color_map,
    labels={'taux_defaut': 'Taux de défaut', 'loan_intent': 'Intention du prêt'},
    title='Taux de défaut par intention — DEBTCONSOLIDATION & MEDICAL en surrisque (orange)'
)
fig8.update_traces(textposition='outside')
fig8.add_hline(y=0.218, line_dash='dot', line_color='white',
               annotation_text='Moyenne 21,8 %', annotation_position='top right')
fig8.update_layout(plot_bgcolor='#021B2E', paper_bgcolor='#021B2E',
                   font_color='white', yaxis_tickformat='.0%', showlegend=False)
fig8.write_html(str(ROOT / 'figures' / 'defaut_intent.html'))
fig8.show()
print("✅ figures/defaut_intent.html sauvegardé")
for _, row in default_intent.iterrows():
    tag = ' ← high_risk_intent=1' if row['loan_intent'] in ['DEBTCONSOLIDATION','MEDICAL'] else ''
    print(f"  {row['loan_intent']:22s} : {row['taux_defaut']:.1%}{tag}")

✅ figures/defaut_intent.html sauvegardé
  DEBTCONSOLIDATION      : 28.6% ← high_risk_intent=1
  MEDICAL                : 26.7% ← high_risk_intent=1
  HOMEIMPROVEMENT        : 26.1%
  PERSONAL               : 19.9%
  EDUCATION              : 17.2%
  VENTURE                : 14.8%


In [20]:
# Validation de l'orthogonalité de high_risk_intent
# r≈0.002 avec debt_service_rate → apport INDÉPENDANT au modèle
from scipy.stats import spearmanr

pairs = [
    ('high_risk_intent', 'debt_service_rate'),
    ('high_risk_intent', 'monthly_payment_proxy'),
    ('high_risk_intent', 'loan_status'),
]
print("Corrélations Spearman de high_risk_intent :")
for col1, col2 in pairs:
    mask = df[[col1, col2]].notna().all(axis=1)
    rho, _ = spearmanr(df.loc[mask, col1], df.loc[mask, col2])
    flag = ' ← ORTHOGONAL ✓' if abs(rho) < 0.05 else ' ← signal propre'
    print(f"  {col1} vs {col2:25s} : r={rho:+.3f}{flag}")

Corrélations Spearman de high_risk_intent :
  high_risk_intent vs debt_service_rate         : r=+0.010 ← ORTHOGONAL ✓
  high_risk_intent vs monthly_payment_proxy     : r=+0.015 ← ORTHOGONAL ✓
  high_risk_intent vs loan_status               : r=+0.101 ← signal propre


---
## Vérification des livrables & Commit GitHub

In [21]:
# Vérification : toutes les figures sont-elles présentes ?
figures_attendues = {
    'hist_income'    : 'Bloc 1.1 — Distribution revenus zoom P99',
    'hist_income_log': 'Bloc 1.1 — Distribution log_income (feature J1S2)',
    'box_home'       : 'Bloc 1.2 — Box plot home_ownership × loan_percent_income',
    'heatmap_corr'   : 'Bloc 2.1 — Corrélations Spearman (num continues seules)',
    'scatter_matrix' : 'Bloc 2.2 — Scatter matrix 4 features pipeline',
    'defaut_home'    : 'Bloc 3.1 — Taux défaut par home_ownership',
    'scatter_features': 'Bloc 3.2 — debt_service_rate vs monthly_payment_proxy',
    'defaut_quartile': 'Bloc 4.1 — Rupture Q1→Q2 (justifie log_income)',
    'defaut_intent'  : 'Bloc 4.2 — Signal high_risk_intent (DEBTCONS/MEDICAL)',
}

print("Vérification des livrables J1S3 :")
print("=" * 65)
all_ok = True
for fig, desc in figures_attendues.items():
    path = ROOT / 'figures' / f'{fig}.html'
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  ✓  {fig:20s}  ({size_kb:>5.0f} KB)  {desc}")
    else:
        print(f"  ✗  {fig:20s}  MANQUANT — réexécuter le bloc correspondant")
        all_ok = False
print("=" * 65)
if all_ok:
    print("\n✅ 9 figures présentes. Prêt pour le commit !")
else:
    print("\n⚠️  Des fichiers manquent.")

# Vérification cohérence pipeline — aucune figure ne doit référencer loan_grade
print("\n✓ Pipeline vérifié : loan_grade absente de tous les graphiques")
print("✓ loan_int_rate présente comme variable source réhabilitée")
print("✓ debt_service_rate = signal Spearman #1 du pipeline")

Vérification des livrables J1S3 :
  ✓  hist_income           ( 4915 KB)  Bloc 1.1 — Distribution revenus zoom P99
  ✓  hist_income_log       ( 5103 KB)  Bloc 1.1 — Distribution log_income (feature J1S2)
  ✓  box_home              ( 5430 KB)  Bloc 1.2 — Box plot home_ownership × loan_percent_income
  ✓  heatmap_corr          ( 4744 KB)  Bloc 2.1 — Corrélations Spearman (debt_service_rate #1)
  ✓  scatter_matrix        ( 4892 KB)  Bloc 2.2 — Scatter matrix 4 features pipeline
  ✓  defaut_home           ( 4743 KB)  Bloc 3.1 — Taux défaut par home_ownership
  ✓  scatter_features      ( 4869 KB)  Bloc 3.2 — debt_service_rate vs monthly_payment_proxy
  ✓  defaut_quartile       ( 4743 KB)  Bloc 4.1 — Rupture Q1→Q2 (justifie log_income)
  ✓  defaut_intent         ( 4745 KB)  Bloc 4.2 — Signal high_risk_intent (DEBTCONS/MEDICAL)

✅ 9 figures présentes. Prêt pour le commit !

✓ Pipeline vérifié : loan_grade absente de tous les graphiques
✓ loan_int_rate présente comme variable source réhabilitée

### Commit Git

```bash
git add notebooks/j1s3_eda_plotly.ipynb figures/
git commit -m "feat(j1s3): EDA Plotly — 9 figures sans loan_grade, debt_service_rate"
git push origin main
```

Vérification :
```bash
git log --oneline -3
```

Résultat attendu :
```
abc1234 feat(j1s3): EDA Plotly — 9 figures sans loan_grade, debt_service_rate
def5678 J1S2: Pandas & NumPy — pipeline sans loan_grade, debt_service_rate
```

---
## ☑ Checklist de fin de session

- ☐ `df.shape = (32581, 15)` au chargement — sans `loan_grade`, avec 4 features J1S2
- ☐ Assertions passées : `loan_grade` absente, `debt_service_rate` présente
- ☐ Piège `log_x=True` compris et évité — `log_income` du parquet utilisé directement
- ☐ 9 figures HTML dans `figures/` — toutes ✓ à la vérification
- ☐ `debt_service_rate` apparaît comme signal #1 sur la heatmap (+0,389)
- ☐ `loan_int_rate` apparaît avec un signal propre (+0,298) — réhabilité sans grade
- ☐ `high_risk_intent` orthogonal confirmé (r≈0,002 vs debt_service_rate)
- ☐ Rupture Q1→Q2 −18,6 pts visible sur le bar chart quartiles
- ☐ Commit pushé — notebook + figures/ visibles sur github.com

---
## ➡ J1S4 — Nettoyage & Feature Engineering

Entrée : `credit_features_j1.parquet` (15 colonnes, sans `loan_grade`)  
Actions : capping P99 (age, emp_length, income), imputation médiane  
(`loan_int_rate` 9,6 %, `person_emp_length` 2,7 %), encodage catégorielles  
Livrable : `credit_risk_clean.parquet` — dataset ML-ready pour tout le Jour 2